# notebooks/04_generation_test.ipynb

In [1]:
%run notebook_init.py

In [2]:
from rag_pipeline.retriever.retrieve import query_chunks
from rag_pipeline.verifier.hallucination_guard import sentence_overlap
from rag_pipeline.llm.generator import generate_answer, format_prompt, generate_answer_with_gate

/Users/aditya/Documents/projects/hallucination-resistant-finance-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dryrun = True # Set to TRUE by default

In [ ]:
def ask_and_verify():
    while True:
        q = input("Ask a question (or 'q' to quit): ")
        if q.lower() == 'q':
            break
        chunks = query_chunks(q)

        print("\n🔎 Retrieved Chunks (Debug Info):")
        for i, c in enumerate(chunks):
            metadata = c.get('metadata', {})
            print(f"\n  Chunk {i+1}:")
            print(f"    doc_id: {metadata.get('doc_id', 'unknown')}")
            print(f"    chunk_id: {metadata.get('chunk_id', 'unknown')}")
            print(f"    section: {metadata.get('section', 'unknown')}")
            print(f"    distance: {c.get('distance', 'N/A')}")
            print(f"    text preview: {c['text'][:200]}...")

        answer, abstained = generate_answer_with_gate(q, chunks, model_name="gpt-4o")
        print("\nAnswer:\n", answer)
        if abstained:
            print("(System abstained from answering)")

In [ ]:
ask_and_verify()

In [ ]:
# Use query_chunks() as single source of truth for retrieval
# This ensures consistent metadata and avoids mismatched Chroma instances

for i, c in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(f"doc_id: {c['metadata'].get('doc_id', 'unknown')}")
    print(f"chunk_id: {c['metadata'].get('chunk_id', 'unknown')}")
    print(f"section: {c['metadata'].get('section', 'unknown')}")
    print(f"distance: {c.get('distance', 'N/A')}")
    print(f"text preview: {c['text'][:300]}...")

# Debug Code Below

In [15]:
from rag_pipeline.retriever.retrieve import query_chunks
from rag_pipeline.llm.generator import generate_answer_with_gate

# Test multiple queries
queries = [
    "Where is Tesla headquartered?",
    "What is Tesla's revenue?",
    # Add more queries here
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print('='*60)
    
    chunks = query_chunks(query, top_k=5)
    answer, abstained = generate_answer_with_gate(query, chunks)
    print(f"================================\nAnswer: {answer}\n=================================")
    print(f"Abstained: {abstained}")


Query: Where is Tesla headquartered?


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.02it/s]


DEBUG: semantic_has_intent=False, semantic_total_chars=12917, substantial_chunks=5/5, use_keyword_fallback=True
DEBUG: entity_keywords=['tesla'], intent_keywords=['headquarters', 'headquartered']
DEBUG: Top keyword matches (score, intent, entity, proximity, length, section_boost):
  [1] ITEM 1A. RISK FACTORS: score=3 (intent=0, entity=1, proximity=False, len=2914, section_boost=2)
  [2] ITEM 1A. RISK FACTORS: score=3 (intent=0, entity=1, proximity=False, len=2856, section_boost=2)
  [3] ITEM 1. BUSINESS: score=3 (intent=0, entity=1, proximity=False, len=2622, section_boost=2)
DEBUG: entity_keywords=['tesla'], intent_keywords=['headquarters', 'headquartered']
DEBUG: Top keyword matches (score, intent, entity, proximity, length, section_boost):
  [1] ITEM 2. PROPERTIES: score=43 (intent=1, entity=1, proximity=True, len=1145, section_boost=2)
  [2] ITEM 1A. RISK FACTORS: score=33 (intent=1, entity=1, proximity=False, len=2906, section_boost=2)
  [3] ITEM 1A. RISK FACTORS: score=33 (intent

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:rag_pipeline:Query: Where is Tesla headquartered?
INFO:rag_pipeline:  [1] tesla-2024-10K::ITEM 2. PROPERTIES::chunk-71
INFO:rag_pipeline:  [2] tesla-2024-10K::ITEM 1A. RISK FACTORS::chunk-61
INFO:rag_pipeline:  [3] tesla-2024-10K::ITEM 1A. RISK FACTORS::chunk-60
INFO:rag_pipeline:  [4] tesla-2024-10K::ITEM 1. BUSINESS::chunk-22
INFO:rag_pipeline:  [5] tesla-2024-10K::ITEM 1A. RISK FACTORS::chunk-40
INFO:rag_pipeline:Answer length: 44 chars


Answer: Tesla is headquartered in Austin, Texas [1].
Abstained: False

Query: What is Tesla's revenue?


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.15it/s]


DEBUG: semantic_has_intent=False, semantic_total_chars=12486, substantial_chunks=5/5, use_keyword_fallback=True
DEBUG: entity_keywords=['tesla', 'revenue', 'what'], intent_keywords=[]
DEBUG: Top keyword matches (score, intent, entity, proximity, length, section_boost):
  [1] ITEM 1A. RISK FACTORS: score=3 (intent=0, entity=1, proximity=False, len=2856, section_boost=2)
  [2] ITEM 1. BUSINESS: score=3 (intent=0, entity=1, proximity=False, len=2622, section_boost=2)
  [3] ITEM 1. BUSINESS: score=3 (intent=0, entity=1, proximity=False, len=2599, section_boost=2)
DEBUG: entity_keywords=['tesla', 'revenue', 'what'], intent_keywords=[]
DEBUG: Top keyword matches (score, intent, entity, proximity, length, section_boost):
  [1] ITEM 1A. RISK FACTORS: score=4 (intent=0, entity=2, proximity=False, len=2892, section_boost=2)
  [2] ITEM 1A. RISK FACTORS: score=4 (intent=0, entity=2, proximity=False, len=2858, section_boost=2)
  [3] ITEM 1A. RISK FACTORS: score=4 (intent=0, entity=2, proximity=Fals

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:rag_pipeline:Query: What is Tesla's revenue?
INFO:rag_pipeline:  [1] tesla-2024-10K::ITEM 1A. RISK FACTORS::chunk-67
INFO:rag_pipeline:  [2] tesla-2024-10K::ITEM 1A. RISK FACTORS::chunk-45
INFO:rag_pipeline:  [3] tesla-2024-10K::ITEM 1A. RISK FACTORS::chunk-59
INFO:rag_pipeline:  [4] tesla-2024-10K::ITEM 1. BUSINESS::chunk-27
INFO:rag_pipeline:  [5] tesla-2024-10K::ITEM 1. BUSINESS::chunk-19
INFO:rag_pipeline:Answer length: 56 chars


Answer: The provided document does not contain that information.
Abstained: False
